# 06 — Train a Vector-DQN Model Online

Train a MOUSE model with **Vector-DQN**: each discrete action is a 2D vector (`vec_dim=2`), rewards rotate the bootstrap target via RoPE, and `VecDqnObjective` minimizes one minus cosine similarity between the taken action vector and the rotated greedy bootstrap.

This notebook mirrors `examples/03_train_online.ipynb` but swaps `DiscreteActionValueHead` + `DqnObjective` for `VectorActionValueHead` + `VecDqnObjective`. Action selection uses angular scores from the two output values per action.

For a quick CPU smoke test, run `examples/run_vec_dqn_learning_demo.py`. For faster iteration here, the backbone is truncated with `num_layers=4`; remove that argument to use the full pretrained `Qwen/Qwen3-0.6B` stack.


In [ ]:
from collections import deque

import torch

from mouse_envs import EnvConfig, make_env
from mouse_core.data import DataLoader, Datastore
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import StepEmbedder
from mouse_core.models.heads import VectorActionValueHead
from mouse_core.objectives import VecDqnObjective


VEC_DIM = 2  # two values per action → direction in the plane


MODEL_ID = "mouse-example-model"  # Hub repo id for push_model_to_hub
MAX_ACTIONS = 4  # discrete action head size (FrozenLake: up/down/left/right)
MAX_OBS_DISCRETE = 64  # observation embedder vocab (max cells in 8×8 maps)
NUM_ENVS = 1000  # parallel SingleEnv instances (one Datastore each)

GRADIENT_STEPS = 20000  # total optimizer updates (same budget as 02_train_offline)
SEQUENCE_LENGTH = 512  # replay sequence length sampled by DataLoader
BATCH_SIZE = 4  # sequences per optimizer step

ENV_STEPS_PER_CYCLE = 1000  # env transitions collected each cycle
STEPS_PER_ENV = 100  # consecutive env steps on one env before rotating
GRADIENT_STEPS_PER_CYCLE = 1000  # optimizer updates each cycle (once learning starts)

LEARNING_STARTS = 2000  # env transitions before the first optimizer update
EXPLORATION_ENDS = 10000  # env step when linear ε decay from 1.0 to 0.0 finishes


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


Online training keeps the environment in the training loop. Each `SingleEnv` instance runs one infinite task with deterministic FrozenLake dynamics and 50-step episode truncation, so the model keeps carrying policy context across episode boundaries.


In [ ]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_online_{i}",
        reset_seed=i,
        episodes_per_task=0,
        kwargs={
            "is_slippery": False,
            "min_width": 4, "max_width": 8,
            "min_height": 4, "max_height": 8,
            "step_penalty": -0.01,
            "max_episode_steps": 50,
            "map_seed": i,
        },
    )
    for i in range(NUM_ENVS)
]

envs = [make_env(config) for config in configs]
print(f"Environments {min(10, len(envs))} of {len(envs)}:")
for env in envs[:10]:
    print(env.name)

## Build the model

Use `VectorActionValueHead` with `vec_dim=2` so each action outputs an `(x, y)` pair. `model.get_action` ranks actions by angular score; `VecDqnObjective` rotates the bootstrap vector by `reward * reward_scale`.

Pass `num_layers=4` to truncate the backbone for a quicker demo run.


In [ ]:
import math

backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B", num_layers=4)

encoder = StepEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1,
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "in_min": 0.01,
            "in_max": 100.0,
            "tokens": 1,
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1,
        },
    ],
    modality_fusion="sum",
    include_type_token=False,
)

head = VectorActionValueHead(
    in_features=backbone.hidden_dim,
    max_num_actions=MAX_ACTIONS,
    vec_dim=VEC_DIM,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device)
print(model)


## Replay buffer and policy contexts

Each env writes to one `Datastore`; together they are the live replay buffer. Each env also has a `contexts` deque capped at `SEQUENCE_LENGTH` — the action-selection history used during collection. Contexts are never cleared; episode boundaries appear as `done` values in the sequence.


In [ ]:
stores = [Datastore(name=env.name) for env in envs]
contexts = [deque(maxlen=SEQUENCE_LENGTH) for _ in envs]


## Rollout phase

One **rollout** gathers up to `ENV_STEPS_PER_CYCLE` env transitions, visiting envs in order and taking up to `STEPS_PER_ENV` consecutive transitions per env before moving on. For each env visit the loop uses a single **B=1 KV cache** (like vLLM per-sequence caching): full-context pass on the first model call, then incremental passes for new rows only. The cache is discarded when moving to the next env — inference is never batched across envs, so only one cache is needed at a time.

1. Choose actions with epsilon-greedy exploration (`epsilon` is recomputed from the global env-step counter on every transition). Random actions skip the model; the next model call feeds all uncached rows as a catch-up slice.
2. Step the env, append to its `Datastore` and `contexts` deque. The KV cache grows incrementally for the duration of that env's visit and is discarded when the loop moves to the next env.

Each rollout clears every env tracker first, then prints mean reward/length for episodes completed during the burst when the rollout finishes.


In [ ]:
def epsilon_for_env_step(env_step: int) -> float:
    """Linear ε decay: 1.0 (explore) → 0.0 (greedy)."""
    if EXPLORATION_ENDS <= 0:
        raise ValueError("EXPLORATION_ENDS must be positive.")
    frac = min(env_step / EXPLORATION_ENDS, 1.0)
    return 1.0 - frac


In [ ]:
def rollout(
    model: Model,
    envs: list,
    stores: list[Datastore],
    contexts: list[deque],
    env_steps: int,
    grad_steps: int,
) -> int:
    """Collect up to ``ENV_STEPS_PER_CYCLE`` env transitions, then return."""
    for env in envs:
        env.tracker.clear()
    model.eval()

    budget = ENV_STEPS_PER_CYCLE
    collected = 0

    for env, store, context in zip(envs, stores, contexts):
        if collected >= budget:
            break

        kv_cache = None
        cache_count = 0
        ctx_count = 0
        chunk = min(STEPS_PER_ENV, budget - collected)

        for _ in range(chunk):
            epsilon = epsilon_for_env_step(env_steps)
            if not context or torch.rand(1).item() < epsilon:
                inp = env.sample_random_input()
            else:
                ctx_list = list(context)
                with torch.no_grad():
                    if kv_cache is None:
                        preds, _, kv_cache = model([ctx_list], use_cache=True)
                    else:
                        uncached = ctx_count - cache_count
                        preds, _, kv_cache = model(
                            [ctx_list[-uncached:]], cache=kv_cache, use_cache=True
                        )
                    cache_count = ctx_count

                action = model.get_action(preds, temperature=0.0, num_actions=MAX_ACTIONS)
                inp = {"action": action.squeeze().cpu()}

            out = env.step(inp)
            store.append({**inp, **out})
            context.append({**inp, **out})
            ctx_count += 1
            env_steps += 1
            collected += 1

    rewards: list[float] = []
    lengths: list[int] = []
    for env in envs:
        rewards.extend(env.tracker.episode_cum_rewards)
        lengths.extend(env.tracker.episode_lengths)
    n = len(rewards)
    mean_r = torch.tensor(rewards).mean().item() if n else float("nan")
    mean_l = torch.tensor(lengths).float().mean().item() if n else float("nan")
    epsilon = epsilon_for_env_step(env_steps)
    print(
        f"env_step={env_steps} grad_step={grad_steps} epsilon={epsilon:.3f}"
        f" | {n} completed episodes | mean reward {mean_r:.3f} | mean length {mean_l:.1f}"
    )
    return env_steps


## Training phase

One **train burst** runs after each rollout once `env_steps >= LEARNING_STARTS`:

1. Call `loader.refresh()` so the replay sampler re-snapshots stores and drops prefetched batches.
2. Run up to `GRADIENT_STEPS_PER_CYCLE` Vector-DQN updates (capped by remaining `GRADIENT_STEPS`) and print loss and action-vector metrics at the end of each burst.

The loader is created up front (stores may still be empty; with `pack=True`, sampling starts once rows have been appended). It uses `pack=True`, so stores shorter than `SEQUENCE_LENGTH` are padded by drawing additional segments. `weight_mode="per_step"` weights sampling toward larger stores. Pass `seed=...` for reproducible batches; default is non-deterministic.


In [ ]:
loader = DataLoader(
    stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    weight_mode="per_step",
    pack=True,
    num_workers=0,
)
objective = VecDqnObjective(
    tau=0.005,
    reward_scale=math.pi,
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1.0e-5,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1.0e-8,
)

In [ ]:
def train_burst(
    model: Model,
    optimizer: torch.optim.Optimizer,
    objective: VecDqnObjective,
    loader: DataLoader,
    env_steps: int,
    grad_steps: int,
    burst_steps: int,
) -> tuple[torch.Tensor, dict[str, float]]:
    """Refresh replay and run ``burst_steps`` optimizer updates."""
    model.train()
    loader.refresh()

    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(burst_steps):
        batch = loader.next_batch()

        predictions, objective_data, _ = model(batch)
        loss, metrics = objective(objective_data, predictions)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.polyak_update(action_vector_tau=objective.tau)

    assert loss is not None
    print(
        f"env_step={env_steps} grad_step={grad_steps + burst_steps}"
        f" | train loss={loss.item():.4f} action_vector={metrics['action_vector']:.4f}"
        f" score_abs={metrics['action_vector_score_abs_mean']:.3f}"
    )
    return loss, metrics


## Run online training

The master loop alternates **rollout → train** until `GRADIENT_STEPS` optimizer updates have been applied — the same total as offline training. Optimizer updates are skipped until `LEARNING_STARTS` env transitions have been collected. Each burst prints stats at the end of `rollout` / `train_burst`.


In [ ]:
env_steps = 0
grad_steps = 0

while grad_steps < GRADIENT_STEPS:
    env_steps = rollout(model, envs, stores, contexts, env_steps, grad_steps)

    if env_steps >= LEARNING_STARTS:
        burst = min(GRADIENT_STEPS_PER_CYCLE, GRADIENT_STEPS - grad_steps)
        train_burst(model, optimizer, objective, loader, env_steps, grad_steps, burst)
        grad_steps += burst

loader.close()

for env in envs:
    env.close()
print(f"Online Vector-DQN finished ({grad_steps} optimizer steps, {env_steps} env steps).")


## Push to the Hub

Run this cell if you want to save the online-trained checkpoint to the shared Hub repo under `MODEL_ID`. The inference notebook loads the current checkpoint from that repo, so whichever training notebook you push last is the model it evaluates.

In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")